In [ ]:
# → 이미 학습된 YOLO + LightGBM 모델로 추론 + 예측결과 저장하는 코드

In [ ]:
# =========================
# 0. 라이브러리 임포트
# =========================
from ultralytics import YOLO  # YOLOv8 모델 로드용
import joblib                 # LightGBM 모델 로드용
import cv2                   # 이미지 처리용
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# =========================
# 1. 모델 로딩
# =========================

# YOLOv8 object detection model (장비, 작업자 탐지용)
yolo_model = YOLO("/kaggle/input/object-detection-model/el_object_detection.pt")

# LightGBM classification model (작업공정 분류용)
gbm_model = joblib.load("/kaggle/input/process-classification-model/el_process_classification.pkl")

In [ ]:
# =========================
# 2. 모든 하위 폴더의 이미지 가져오기
# =========================

import glob

# 이미지가 들어 있는 루트 폴더 (Training 원천데이터 기준)
img_root = "/kaggle/input/workprocess-detection/Training/01.원천데이터"

# 하위 폴더 포함 모든 이미지 파일 가져오기 (.jpg 기준)
image_paths = glob.glob(os.path.join(img_root, "**", "*.jpg"), recursive=True)

print(f"총 {len(image_paths)}장의 이미지가 발견되었습니다.")


In [ ]:
# =========================
# 3. 이미지 하나씩 처리 시작
# =========================

for img_path in image_paths:
    # 이미지 로딩
    img = cv2.imread(img_path)
    if img is None:
        print(f"[경고] 이미지 불러오기 실패: {img_path}")
        continue

    # YOLOv8 예측
    results = yolo_model.predict(source=img, conf=0.4, verbose=False)

    for r in results:
        for box in r.boxes:
            # 클래스 ID, confidence score
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = yolo_model.names[cls_id]

            # 바운딩 박스 좌표
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cropped = img[y1:y2, x1:x2]

            # 예외처리: 박스가 이미지 바깥을 벗어나는 경우
            if cropped.size == 0:
                continue

            # =====================
            # 4. 특징 추출 (RGB 평균)
            # =====================
            mean_rgb = cv2.mean(cropped)[:3]  # (B, G, R)
            feature = np.array(mean_rgb[::-1]).reshape(1, -1)  # → (R, G, B)

            # =====================
            # 5. LightGBM 분류
            # =====================
            predicted_class = gbm_model.predict(feature)[0]

            # =====================
            # 6. 출력 및 시각화
            # =====================
            print(f"{os.path.basename(img_path)} → [{label}] (conf: {conf:.2f}) → 작업공정 분류: {predicted_class}")

            # 바운딩 박스 시각화
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, f"{label}/{predicted_class}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


# 예측 결과 누적 저장
import pandas as pd
results_list = []

# 루프 안에서:
results_list.append({
    "image": os.path.basename(img_path),
    "bbox_class": label,
    "conf": conf,
    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
    "predicted_process": predicted_class
})

# 루프 끝난 후 CSV 저장
df = pd.DataFrame(results_list)
df.to_csv("yolo_lgbm_predictions.csv", index=False)